# Treinamento temporal dos modelos de arboviroses

- Selecione exatamente uma doença no bloco de configuração.
- Dengue: treino 2017–2019, validação 2020 e teste lacrado 2021.
- Chikungunya: treino 2017–2023, validação 2024 e teste lacrado 2025.

A MLP ajusta encoder e medianas somente no treino. XGBoost roda na GPU (`device="cuda"`) e LightGBM em CPU; ambos preservam `NaN` nativamente. O fit final usa todo o treino; o Optuna afina numa amostra estratificada.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('../..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from dengue_pipeline.diseases import get_disease_config, select_disease


DENGUE              = 0                    # 1 para treinar dengue
CHIKUNGUNYA         = 1                    # 1 para treinar chikungunya
DISEASE             = select_disease(DENGUE, CHIKUNGUNYA)
DISEASE_CONFIG      = get_disease_config(DISEASE)

N_TRIALS            = 200                  # trials de Optuna (XGBoost/LightGBM)
TUNING_SAMPLE_SIZE  = 400_000             # amostra estratificada para o tuning
MAX_EPOCHS          = 150                 # teto de épocas da MLP (early stopping corta antes)

MLP_HIDDEN          = "1024,512,256,128"  # larguras das camadas ocultas (vírgula separa)
MLP_LEARNING_RATE   = 1e-4
MLP_BATCH_SIZE      = 16_384
MLP_DROPOUT         = 0.2
MLP_PATIENCE        = 10

DEVICE              = "cuda"              # MLP e XGBoost: "cuda" ou "cpu"
LGBM_DEVICE         = "cpu"               # LightGBM: "cpu" ou "gpu" (GPU nem sempre é mais rápido)

print("Doença:", DISEASE)
print("Split temporal:", DISEASE_CONFIG.train_years, "| val", DISEASE_CONFIG.validation_years, "| teste", DISEASE_CONFIG.test_years)

In [ ]:
# Monta o comando com as variáveis do cell de config e dispara o treino.
cmd = (
    f"../../scripts/train_models.py --disease {DISEASE} "
    f"--n-trials {N_TRIALS} --tuning-sample-size {TUNING_SAMPLE_SIZE} --max-epochs {MAX_EPOCHS} "
    f"--mlp-hidden {MLP_HIDDEN} --mlp-learning-rate {MLP_LEARNING_RATE} "
    f"--mlp-batch-size {MLP_BATCH_SIZE} --mlp-dropout {MLP_DROPOUT} --mlp-patience {MLP_PATIENCE} "
    f"--device {DEVICE} --lgbm-device {LGBM_DEVICE}"
)
print("Executando:", cmd)
get_ipython().run_line_magic("run", cmd)